In [8]:
import os
import torch
import argparse

import shared.utils as su
from models.qwen3vl_embedding import Qwen3VLEmbedder
from tasks.eval_chiral_retrieval import load_data
from utils.chiral_retrieval_metrics import (
    compute_metrics,
    print_metrics_as_latex_row,
)
import pandas as pd
import json

In [13]:
def gather_text_features(model, texts):
    """Compute text embeddings for all unique text IDs."""
    texts_feat = {}
    for text in enumerate(
        su.log.tqdm_iterator(texts, desc='Computing text features')
    ):
        emb = model.process([{'text': text}])
        zt = emb.squeeze(0).cpu().float()
        texts_feat[text] = zt
        if j == 0:
            print("Text embedding:", zt.shape)
    return texts_feat


def gather_video_features(
    df,
    model,
    # fps,
    # max_frames,
):
    """Compute video embeddings for all unique video IDs."""
    video_ids = df.video_path.unique()
    video_feat = {}
    for j, video_id in enumerate(
        su.log.tqdm_iterator(video_ids, desc='Computing video features')
    ):
        video_path = df[df.video_path == video_id].video_path.unique()[0]
        emb = model.process([{
            'video': video_path,
            # 'fps': fps,
            # 'max_frames': max_frames,
            'nframes': 16,
        }])
        zv = emb.squeeze(0).cpu().float()
        video_feat[video_id] = zv
        if j == 0:
            print("Video embedding:", zv.shape)
    return video_feat

In [3]:
# model_path = "/work/piyush/pretrained_checkpoints/Qwen3-VL-Embedding-8B"
model_path = "/work/piyush/experiments/CaRe/Qwen3-VL-Embedding-8B/final-10112025/nli_9000+ego_1000+subj_replaced-seed_42/"

model = Qwen3VLEmbedder(
    model_name_or_path=model_path,
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
    device_map="cuda:0",
)
su.misc.num_params(model.model)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

::: Number of total parameters in Qwen3VLForEmbedding: 8144.794M


### Classification

In [6]:
def load_data_video_cls(
    data_root='/scratch/shared/beegfs/piyush/datasets/MMEB-V2',
    cfg_path='/users/piyush/projects/VLM2Vec/experiments/public/eval/video_cls.yaml'
):
    # Load meta config
    meta_config = su.io.load_yml(cfg_path)

    # Generate dataframe of video paths
    df_video = []
    for ds_key in su.log.tqdm_iterator(meta_config, desc='Gathering video paths'):
        file_name = meta_config[ds_key]['json_name']
        data_file = f'{data_root}/video-tasks/data/{file_name}'
        assert os.path.exists(data_file)
        data = su.io.load_jsonl(data_file)

        ds_name = os.path.basename(meta_config[ds_key]['frame_root'])
        for d in data:
            video_id = d['video_id']
            video_dir = f"{data_root}/video-tasks/frames/{ds_name}/{video_id}"
            assert os.path.isdir(video_dir)
            df_video.append(
                dict(ds_key=ds_key, ds_name=ds_name, video_id=video_id, video_dir=video_dir)
            )
    df_video = pd.DataFrame(df_video)
    assert len(df_video.video_id.unique()) == len(df_video)
    return df_video


def load_data_video_ret(
    data_root = '/scratch/shared/beegfs/piyush/datasets/MMEB-V2',
    cfg_path = '/users/piyush/projects/VLM2Vec/experiments/public/eval/video_ret.yaml',
    video_root = "/scratch/shared/beegfs/piyush/datasets/MMEB-V2/video-tasks/frames/data/ziyan/video_retrieval"
):
    # Load meta config
    meta_config = su.io.load_yml(cfg_path)

    # Load video root
    # This defines the huggingface repo and subset for each dataset
    # (repo, subset, split)
    json_paths = {
        "MSR-VTT": ("VLM2Vec/MSR-VTT", "test_1k", "test"),
        "MSVD": ("VLM2Vec/MSVD", None, "test"),
        "DiDeMo": ("VLM2Vec/DiDeMo", None, "test"),
        # "YouCook2": ("VLM2Vec/YouCook2", None, "val"), # HF version compatibility issue
        "YouCook2": ("lmms-lab/YouCook2", None, "val"),
        "VATEX": ("VLM2Vec/VATEX", None, "test"),
    }

    video_id_extractor = {
        "MSR-VTT": lambda x: x['video_id'],
        "MSVD": lambda x: x['video_id'],
        "DiDeMo": lambda x: x['video'].split('/')[-1].split('.')[0],
        "YouCook2": lambda x: x['id'],
        "VATEX": lambda x: x['videoID'],
    }


    df_video = {'video_id': [], 'video_dir': []}
    from datasets import load_dataset
    for ds_key in su.log.tqdm_iterator(meta_config, desc='Processing datasets'):
        print(ds_key)
        repo, subset, split = json_paths[ds_key]
        df = pd.DataFrame(load_dataset(repo, subset)[split])
        video_dir = f"{video_root}/{ds_key}/frames"
        video_ids = os.listdir(video_dir)
        assert len(video_ids) == len(df)
        # print(df.iloc[0])
        
        df['video_id'] = df.apply(lambda x: video_id_extractor[ds_key](x), axis=1)
        df['video_dir'] = df['video_id'].apply(lambda x: f"{video_root}/{ds_key}/frames/{x}")
        df_video['video_id'].extend(df['video_id'].tolist())
        df_video['video_dir'].extend(df['video_dir'].tolist())
        print('-' * 100)
    df_video = pd.DataFrame(df_video)
    assert len(df_video.video_id.unique()) == len(df_video)
    return df_video

In [9]:
# from evals_tarsier2.compute_embeddings_mmeb import (
#     load_data_video_cls,
#     load_data_video_ret,
# )

df = load_data_video_cls()
df['video_path'] = df['video_dir'].apply(lambda x: x.replace('video-tasks/frames', 'video-tasks/videos') + '.mp4')
df = df[df['video_path'].apply(os.path.exists)]
len(df)

Gathering video paths:   0%|          | 0/5 [00:00<?, ?it/s]

4433

In [14]:
video_feat = gather_video_features(df, model)
len(video_feat)

Computing video features:   0%|          | 0/4433 [00:00<?, ?it/s]

qwen-vl-utils using torchcodec to read video.


Video embedding: torch.Size([4096])


KeyboardInterrupt: 

In [ ]:
save_dir = "/scratch/shared/beegfs/piyush/datasets/MMEB-V2/features"
task = "cls"
model_name = "qwen3vlembedding-finetuned"
save_name = f"{model_name}_video_embeddings_mmebv2_video_{task}.pt"
os.makedirs(save_dir, exist_ok=True)
torch.save(video_embeddings, os.path.join(save_dir, save_name))
print(f"Saved video embeddings to {os.path.join(save_dir, save_name)}")

### Retrieval

In [ ]:
df = load_data_video_ret()
df['video_path'] = df['video_dir'].apply(lambda x: x.replace('video-tasks/frames', 'video-tasks/videos') + '.mp4')
df = df[df['video_path'].apply(os.path.exists)]
len(df)

In [ ]:
video_feat = gather_video_features(df, model)
len(video_feat)

In [ ]:
save_dir = "/scratch/shared/beegfs/piyush/datasets/MMEB-V2/features"
task = "ret"
model_name = "qwen3vlembedding-finetuned"
save_name = f"{model_name}_video_embeddings_mmebv2_video_{task}.pt"
os.makedirs(save_dir, exist_ok=True)
torch.save(video_embeddings, os.path.join(save_dir, save_name))
print(f"Saved video embeddings to {os.path.join(save_dir, save_name)}")